# Expérimentation de la méthode d'évaluation "Retrieval"

## Importation des bibliothèques

In [1]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import pytrec_eval
import json

/info/etu/m2/s2101703/miniconda3/envs/ETE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-06 12:02:08.518917: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 12:02:08.570978: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 12:02:11.931542: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. Y

## Chargement du corpus et du modèle

In [2]:
newsgroups = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))
docs = newsgroups['data']

topic_model = BERTopic(calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs)

## Retrieval

In [3]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,\n\nI am sure some bashers of Pens fans are pr...,0,0_game_team_games_he,"[game, team, games, he, players, season, hocke...","[Scoring stats for the Swedish NHL players, Ap...",game - team - games - he - players - season - ...,0.259929,False
1,My brother is in the market for a high-perform...,5,5_card_monitor_video_drivers,"[card, monitor, video, drivers, vga, monitors,...",[Hello all.\n\tI am thinking about buying an e...,card - monitor - video - drivers - vga - monit...,0.180635,False
2,\n\n\n\n\tFinally you said what you dream abou...,-1,-1_to_the_of_and,"[to, the, of, and, is, for, in, you, that, it]",[Archive-name: x-faq/part4\nLast-modified: 199...,to - the - of - and - is - for - in - you - th...,0.764806,False
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,6,6_drive_scsi_drives_ide,"[drive, scsi, drives, ide, disk, controller, h...",[Thanks to all who responded to my original po...,drive - scsi - drives - ide - disk - controlle...,0.170453,False
4,1) I have an old Jasmine drive which I cann...,104,104_tape_backup_drive_tapes,"[tape, backup, drive, tapes, wangdat, munroe, ...",[I have a Colorado Memory Systems Jumbo 250 ta...,tape - backup - drive - tapes - wangdat - munr...,0.280776,False
...,...,...,...,...,...,...,...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...,12,12_medical_hiv_doctor_patients,"[medical, hiv, doctor, patients, cancer, disea...",[The biggest reason why the cost of medical ca...,medical - hiv - doctor - patients - cancer - d...,0.203470,False
18842,\nNot in isolated ground recepticles (usually ...,175,175_ground_grounding_conductor_neutral,"[ground, grounding, conductor, neutral, wire, ...","[\nNot according to the NEC nor the CEC, as ex...",ground - grounding - conductor - neutral - wir...,0.565626,False
18843,I just installed a DX2-66 CPU in a clone mothe...,85,85_fan_cpu_heat_sink,"[fan, cpu, heat, sink, fans, cooling, chip, ho...","[N(P>Just got a 66MHz 486DX2 system, and am co...",fan - cpu - heat - sink - fans - cooling - chi...,0.626998,False
18844,\nWouldn't this require a hyper-sphere. In 3-...,18,18_den_polygon_points_algorithm,"[den, polygon, points, algorithm, xxxx, plane,...","[\nSorry!! :-)\n\nCall the four points A, B, C...",den - polygon - points - algorithm - xxxx - pl...,0.224115,False


In [4]:
def getWordsTopics(topicsKey :int, model : BERTopic) -> list :
    return [word for word,_ in model.get_topic(topicsKey)]

In [5]:
def prepare_data(topic_model: BERTopic) -> tuple[dict, dict]:
    """
    Prépare le dictionnaire qrel et run pour la bibliothèque pytrec_eval
    topic_model: modèle BERTopic
    return: dictionnaire qrel et run
    """
    
    documentInfos = topic_model.get_document_info(docs=docs)
    topicKeys = topic_model.get_topics().keys()
    qrel = dict()
    predictions = dict()

    print(list(topicKeys)[1:2])
    for i, topic in enumerate(list(topicKeys)[1:2]):
        topic_id = str(topic)
        words = getWordsTopics(topic, topic_model)
        print(words)
        qrel[topic_id] = {}
        predictions[topic_id] = {}
        embeddingMotsCles = topic_model.embedding_model.embed_words(words=words)
        for j, doc in tqdm(documentInfos.iterrows()):
            if doc['Topic'] != -1:
                text = doc['Document']
                embeddingDoc = topic_model.embedding_model.embed_documents(document=[text])
                doc_id = str(j)
                predictions[topic_id][doc_id] = float(cosine_similarity(embeddingMotsCles, embeddingDoc)[0][0])
                if topic == doc['Topic']:
                    qrel[topic_id][doc_id] = 1
                else:
                    qrel[topic_id][doc_id] = 0
    return qrel, predictions

In [6]:
def retrieval(topic_model: BERTopic) -> {}:
    """
    Calcule les métriques de "retrieval" en préparant le dictionnaire qrel et run à l'aide de la bibliothèque pytrec_eval.
    Métriques : 
        - Mean Average Precision (MAP)
        - Normalized Discounted Cumulative Gain (NDCG)
    topic_model: modèle BERTopic
    return: dictionnaire des métriques
    """
    
    qrel, predictions = prepare_data(topic_model=topic_model)

    evaluator = pytrec_eval.RelevanceEvaluator(qrel, {'map', 'ndcg'})
    results = evaluator.evaluate(predictions)
    
    return results

In [7]:
results = retrieval(topic_model=topic_model)

[0]
['game', 'team', 'games', 'he', 'players', 'season', 'hockey', 'play', '25', 'year']


18846it [01:13, 257.15it/s]


In [9]:
results

{'0': {'map': 0.3289907125099784, 'ndcg': 0.8197755977107369}}